# Human Games

This notebook runs through the data from the humans playing the wikipedia game.
This is processed similarly to the ML agents but there are slight differences with how the data needs to be loaded.

In [ ]:
from __future__ import annotations

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import nbinom

from agents.evaluate import step_hist

main_dir = os.path.dirname(os.getcwd())

In [ ]:
human_games = pd.read_csv(
    os.path.join(main_dir, "data/human/human_game_data.tsv"), sep="\t"
)

In [ ]:
human_games.head(5)

In [ ]:
success, success_err = step_hist.faction_success(human_games["Success?"] == "Yes")

print(f"The success rate of the human players is: {success:.2%} +/- {success_err:.2%}")

In [ ]:
(mean, mean_std), (spread, spread_std) = step_hist.bootstrap_steps(
    human_games["Steps"][human_games["Success?"] == "Yes"].values
)

print(
    f"The mean number of steps taken by the human players is:"
    f" {mean:.2f} +/- {mean_std:.2f}"
)
print(
    f"The spread of the number of steps taken by the human players is:"
    f" {spread:.2f} +/- {spread_std:.2f}"
)

In [ ]:
(n, n_std), (p, p_std) = step_hist.fit_nbinom(
    human_games["Steps"][human_games["Success?"] == "Yes"].values,
    human_games["OptimalSteps"][human_games["Success?"] == "Yes"].values,
)

print(
    f"The negative binomial fit to the number of steps taken by the human players is:"
    f" n = {n:.2f} +/- {n_std:.2f}, p = {p:.3f} +/- {p_std:.3f}"
)

In [ ]:
fig, ax = plt.subplots()

x_vals = np.arange(0, 100, 1)

ax.plot(
    x_vals,
    nbinom.pmf(x_vals, n, p),
    "-k",
    label=f"Negative Binomial Fit: n = {n:.2f}, p = {p:.3f}",
)

ax.hist(
    (human_games["Steps"] - human_games["OptimalSteps"])[
        human_games["Success?"] == "Yes"
    ].values,
    bins=range(0, 100, 1),
    density=True,
    color="red",
    alpha=0.8,
)

ax.set_xlim(0, 40)

ax.set_xlabel("Number of steps above optimal")
ax.set_ylabel("Fraction of successful games")

ax.grid(True)
ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)
ax.xaxis.set_minor_locator(plt.MultipleLocator(2))
ax.yaxis.set_minor_locator(plt.MultipleLocator(0.01))

We can see that the negative binomial distribution is a good fit for the data with a path size (n) of 3.2 which is much higher than the ML agents which hover around 1. This is closer to waht one would expect if there was an ideal path in mind that the human players were trying to follow.

In [ ]:
def string_to_seconds(time_str):
    minutes, seconds = time_str.split(":")
    return int(minutes) * 60 + int(seconds)


human_games["Seconds"] = human_games["Time (m:s)"][
    human_games["Success?"] == "Yes"
].map(string_to_seconds)

human_games["Seconds"].describe()

In [ ]:
fig, ax = plt.subplots()

ax.hist(
    human_games["Seconds"],
    bins=range(0, 600, 30),
    density=True,
    color="red",
    alpha=0.8,
)

ax.set_xlim(0, 600)

ax.set_xlabel("Time taken to solve the game (s)")
ax.set_ylabel("Fraction of successful games")

ax.grid(True)
ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)
ax.xaxis.set_major_locator(plt.MultipleLocator(60))
ax.xaxis.set_minor_locator(plt.MultipleLocator(30))
ax.yaxis.set_minor_locator(plt.MultipleLocator(0.01))

The median time for the games what will be successful within 10 minutes is only slightly above three minutes, and there is a reasonable drop off at higher times. 
